In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from vllm import LLM,SamplingParams
from vllm_utils import *
from utils import *
from ei_config import EIConfig
from sft_trainer import SFTTrainer,SFTDataset,EIDataset
import os
os.chdir('../')
os.getcwd()

INFO 03-07 13:21:27 __init__.py:190] Automatically detected platform cuda.


'/home/shared/ml/assignment5-alignment'

In [2]:
config_path = 'cs336_alignment/configs/ei_math.json'
config = EIConfig.from_json(config_path)
config.sampling_params

✅ EIConfig loaded from cs336_alignment/configs/ei_math.json


SamplingParams(n=4, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.8, top_p=0.9, top_k=-1, min_p=0.0, seed=42, stop=['</answer>'], stop_token_ids=[], bad_words=[], include_stop_str_in_output=True, ignore_eos=False, max_tokens=1024, min_tokens=32, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None)

In [3]:
model_name ='model/Qwen2.5-Math-1.5B'
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map=get_device(1),# device 1
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
vllm = init_vllm(model_name,device=get_device(2),seed=config.seed,gpu_memory_utilization=0.5)

INFO 03-07 13:21:48 config.py:542] This model supports multiple tasks: {'generate', 'embed', 'reward', 'classify', 'score'}. Defaulting to 'generate'.
INFO 03-07 13:21:48 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda:2, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=model/Qwen2.5-Math-1.5B, num_schedul

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-07 13:21:52 model_runner.py:1115] Loading model weights took 0.0000 GB
INFO 03-07 13:21:54 worker.py:267] Memory profiling takes 1.17 seconds
INFO 03-07 13:21:54 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.50) = 11.79GiB
INFO 03-07 13:21:54 worker.py:267] model weights take 0.00GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 0.00GiB; the rest of the memory reserved for KV Cache is 11.79GiB.
INFO 03-07 13:21:54 executor_base.py:110] # CUDA blocks: 27600, # CPU blocks: 9362
INFO 03-07 13:21:54 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 107.81x
INFO 03-07 13:22:01 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:26<00:00,  1.33it/s]

INFO 03-07 13:22:27 model_runner.py:1562] Graph capturing finished in 26 secs, took 0.00 GiB
INFO 03-07 13:22:27 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 34.94 seconds


In [5]:
class EITrainer:
    def __init__(
        self,
        model: AutoModelForCausalLM,
        tokenizer : AutoTokenizer,
        optimizer : torch.optim.Optimizer,
        config: EIConfig, 
        vllm: LLM,
    ):
        # 模型和优化器配置
        self.model = model
        self.tokenizer = tokenizer
        self.optimizer = optimizer
        self.config =  config
        
        # vllm：离线推理模型
        self.vllm = vllm
        self.ei_dataset =  EIDataset(
            vllm = vllm,
            sampling_params = self.config.sampling_params,
            reward_fn = self.config.reward_fn,
            json_path = self.config.train_dataset_path,
            prompt_template_path=self.config.prompt_template_path,
            sft_sample_size = self.config.sft_sample_size
        )
    def train(self):
        for iteration in range(self.config.ei_iterations):
            load_policy_into_vllm_instance(self.model,self.vllm)# \theta_old
            rollout_prompts, rollout_ground_truths, rollout_answers = self.ei_dataset.get_ei_batch()# sample_responses -> vllm evaluate
            
            print(f"Iteration {iteration+1}: obtained {len(rollout_prompts)} valid samples for SFT training.")

            # 从现有prompt和ground_truths中构建一个新的SFT数据集
            train_dataset = SFTDataset.from_prompts_and_ground_truths(
                prompts=rollout_prompts,
                ground_truths=rollout_ground_truths,
                answers=rollout_answers,
            )

            sft_trainer = SFTTrainer.from_ei_trainer(
                ei_trainer=self,
                train_dataset=train_dataset,
            )
            
            sft_trainer.train()

            # 显示调用save_checkpoint. train函数中save_interval设置为无穷大
            os.makedirs(self.config.save_dir, exist_ok=True)
            save_it_dir = os.path.join(self.config.save_dir,f'EI_iteration_{iteration+1}')
            os.makedirs(save_it_dir, exist_ok=True)
            sft_trainer.save_checkpoint(save_path=save_it_dir)

验证以下函数的功能：
+ from_ei_trainer

In [6]:
ei_trainer = EITrainer(
    model=model,
    tokenizer=tokenizer,
    optimizer=torch.optim.AdamW(model.parameters(), lr=config.max_lr),
    config=config,
    vllm=vllm,   
)

In [7]:
ei_dataset = EIDataset(
    vllm = vllm,
    sampling_params = config.sampling_params,
    reward_fn = config.reward_fn,
    json_path = config.train_dataset_path,
    prompt_template_path=config.prompt_template_path,
    sft_sample_size = config.sft_sample_size
)

In [8]:
rollout_prompts, rollout_ground_truths, rollout_answers = ei_dataset.get_ei_batch()
rollout_prompts[0], rollout_ground_truths[0],rollout_answers[0], len(rollout_answers)

Processed prompts:  25%|██▌       | 128/512 [01:01<03:03,  2.09it/s, est. speed input: 349.72 toks/s, output: 2765.87 toks/s]


('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Expand the following expression: $(13x+15)\\cdot 2x$\nAssistant: <think>',
 'The expression can be expanded by distributing the 2x across the terms inside the parentheses. This gives us: $2x \\cdot 13x + 2x \\cdot 15$</think> <answer>$26x^2 + 30x$</answer>',
 '26x^2+30x',
 41)

In [12]:
train_dataset = SFTDataset.from_prompts_and_ground_truths(
    prompts=rollout_prompts,
    ground_truths=rollout_ground_truths,
    answers=rollout_answers,
)
train_dataset[0]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Expand the following expression: $(13x+15)\\cdot 2x$\nAssistant: <think>',
 'The expression can be expanded by distributing the 2x across the terms inside the parentheses. This gives us: $2x \\cdot 13x + 2x \\cdot 15$</think> <answer>$26x^2 + 30x$</answer>')

In [13]:
sft_trainer = SFTTrainer.from_ei_trainer(
    ei_trainer=ei_trainer,
    train_dataset=train_dataset, 
)

In [16]:
len(sft_trainer.train_dataset),len(sft_trainer.test_dataset)

(41, 500)